# Annexe — Nt2 Analyse de reseaux Telegram

Ce notebook constitue le second volet de l’annexe consacrée à l’analyse de réseau menée sur l’ensemble du corpus Telegram téléchargé, composé de 151 espaces de communication. Contrairement aux analyses thématiques du mémoire, qui se concentrent ensuite principalement sur les chats, cette annexe travaille sur l’ensemble des espaces disponibles afin d’observer les circulations entre eux.

L’objectif est de transformer les exports JSON Telegram en fichiers exploitables dans Gephi. Le réseau est construit à partir des messages transférés : chaque espace de communication devient un nœud, et chaque transfert identifié depuis un autre espace du corpus devient un lien orienté entre deux nœuds. Cette approche permet d’observer quels espaces relaient des contenus provenant d’autres espaces et comment l’information circule à l’intérieur du corpus téléchargé.


## 1. Préparation de l’environnement et localisation des fichiers

Cette première partie importe les bibliothèques nécessaires, définit le dossier contenant les exports JSON Telegram, puis liste les fichiers disponibles. Elle permet de vérifier que le corpus local utilisé pour l’analyse de réseau correspond bien à l’ensemble des espaces téléchargés.


In [18]:
import os
import json
import pandas as pd
from tqdm.notebook import tqdm
import gc


In [19]:
folder_path = "/Users/quentinnippert/Documents/mm_files/full_dataset_jsons"

In [20]:
json_files = [
    f for f in os.listdir(folder_path)
    if f.endswith(".json") and os.path.isfile(os.path.join(folder_path, f))
]

print(f"JSON files found: {len(json_files)}")

JSON files found: 151


## 2. Création d’une table de métadonnées des espaces

Cette section parcourt chaque fichier JSON afin d’extraire les informations générales de chaque espace : nom, type, identifiant Telegram et nom du fichier source. Ces informations servent ensuite à construire la table des nœuds du réseau.

L’identifiant Telegram est normalisé sous la forme `channel...`, car c’est ce format qui apparaît dans le champ `forwarded_from_id` des messages transférés. Cette normalisation est indispensable pour pouvoir faire correspondre les sources des transferts aux espaces présents dans le corpus.


In [ ]:
metadata_rows = []
errors_meta = []

for filename in tqdm(json_files, desc="Читаю metadata"):
    file_path = os.path.join(folder_path, filename)

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        raw_id = data.get("id")

        metadata_rows.append({
            "filename": filename,
            "json_name": data.get("name"),
            "json_type": data.get("type"),
            "json_id_raw": raw_id,
            "telegram_id": f"channel{raw_id}" if raw_id is not None else None
        })

        del data
        gc.collect()

    except Exception as e:
        errors_meta.append((filename, str(e)))

df_meta = pd.DataFrame(metadata_rows)

print(f"Metadata read: {len(df_meta)}")
print(f"Errors in metadata: {len(errors_meta)}")

df_meta.head()

In [23]:
print(f"Total JSON: {len(df_meta)}")
print(f"Unique telegram_id: {df_meta['telegram_id'].nunique()}")
print(f"Empty telegram_id: {df_meta['telegram_id'].isna().sum()}")

Total JSON: 151
Unique telegram_id: 151
Empty telegram_id: 0


In [24]:
duplicate_ids = (
    df_meta["telegram_id"]
    .value_counts(dropna=False)
)

duplicate_ids = duplicate_ids[duplicate_ids > 1]

print(f"Duplicate telegram_id: {len(duplicate_ids)}")

df_meta[df_meta["telegram_id"].isin(duplicate_ids.index)].sort_values("telegram_id")

Duplicate telegram_id: 0


,filename,json_name,json_type,json_id_raw,telegram_id


## 3. Préparation des dictionnaires de correspondance

À partir de la table de métadonnées, plusieurs dictionnaires sont créés pour retrouver rapidement le nom, le fichier et le type d’un espace à partir de son identifiant Telegram. Ces dictionnaires permettent ensuite de transformer les identifiants techniques des messages transférés en informations lisibles pour l’analyse.


In [25]:
#Dictionnaires for fast matching 

all_telegram_ids = set(df_meta["telegram_id"].dropna())

id_to_name = dict(zip(df_meta["telegram_id"], df_meta["json_name"]))
id_to_filename = dict(zip(df_meta["telegram_id"], df_meta["filename"]))
id_to_type = dict(zip(df_meta["telegram_id"], df_meta["json_type"]))

print(f"ID for matching: {len(all_telegram_ids)}")

ID for matching: 151


In [27]:
id_to_type

{'channel1700758525': 'public_supergroup',
 'channel1707130413': 'private_supergroup',
 'channel1673172889': 'public_channel',
 'channel1623635238': 'public_supergroup',
 'channel1670808426': 'private_supergroup',
 'channel1702859286': 'public_channel',
 'channel1643805351': 'public_supergroup',
 'channel1723248257': 'public_channel',
 'channel1423405867': 'private_supergroup',
 'channel1539747708': 'public_supergroup',
 'channel1689990766': 'public_supergroup',
 'channel1344147431': 'public_supergroup',
 'channel1688077858': 'public_channel',
 'channel1792058402': 'public_supergroup',
 'channel1539434800': 'public_channel',
 'channel1687942752': 'public_supergroup',
 'channel1705142012': 'public_channel',
 'channel1504072664': 'public_channel',
 'channel1591427421': 'private_supergroup',
 'channel1792739729': 'private_supergroup',
 'channel1765433763': 'public_channel',
 'channel1273471778': 'public_supergroup',
 'channel1691851482': 'public_supergroup',
 'channel1877139886': 'public_

## 4. Extraction des messages transférés entre espaces du corpus

Cette partie constitue le cœur de la préparation du réseau. Le notebook parcourt tous les messages de chaque export JSON et recherche le champ `forwarded_from_id`. Lorsqu’un message a été transféré depuis un autre espace dont l’identifiant est présent dans le corpus, une relation est enregistrée.

Chaque ligne de la table produite correspond à un transfert repéré : elle contient l’espace source, l’espace cible, le fichier concerné, l’identifiant du message et sa date. Le lien est orienté de l’espace d’origine du message vers l’espace dans lequel le message transféré a été retrouvé.


In [ ]:
edges = []
errors_messages = []

for filename in tqdm(json_files, desc="Searching for forwarded_from_id"):
    file_path = os.path.join(folder_path, filename)

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        target_raw_id = data.get("id")
        target_id = f"channel{target_raw_id}" if target_raw_id is not None else None
        target_name = data.get("name")

        messages = data.get("messages", [])

        for message in messages:
            forwarded_from_id = message.get("forwarded_from_id")

            if forwarded_from_id in all_telegram_ids:
                edges.append({
                    "source_id": forwarded_from_id,
                    "source_name": id_to_name.get(forwarded_from_id),
                    "source_file": id_to_filename.get(forwarded_from_id),

                    "target_id": target_id,
                    "target_name": target_name,
                    "target_file": filename,

                    "message_id": message.get("id"),
                    "date": message.get("date"),
                    "date_unixtime": message.get("date_unixtime"),

                    "forwarded_from": message.get("forwarded_from")
                })

        del data
        gc.collect()

    except Exception as e:
        errors_messages.append((filename, str(e)))

df_edges_raw = pd.DataFrame(edges)

print(f"Found forwarded messages with matching ID: {len(df_edges_raw)}")
print(f"Errors reading messages: {len(errors_messages)}")

df_edges_raw.head()

In [29]:
df_edges_raw.to_csv("/Users/quentinnippert/Documents/mm_files/Telegram_network/edges_raw.csv", index=False, encoding="utf-8")

## 5. Enrichissement des métadonnées avec la table du corpus

Cette section ajoute aux métadonnées issues des fichiers JSON les informations qualitatives présentes dans la table de suivi du corpus, notamment le titre du support, la ville ou région associée et la thématique. Ces variables sont utiles pour visualiser le réseau dans Gephi et colorer ou filtrer les nœuds selon leurs caractéristiques.


In [33]:
df = pd.read_csv('corpus_telegram_avril2026 - Sheet1.csv')

In [35]:
df_support = df.copy()

# На всякий случай оставляем только нужные колонки
cols_needed = [
    "Titre_support",
    "Ville/Région",
    "Thématique"
]

df_support = df_support[cols_needed].copy()

# Если в таблице есть повторы по Titre_support, оставляем первый вариант
df_support = df_support.drop_duplicates(subset=["Titre_support"])

In [36]:
df_support

,Titre_support,Ville/Région,Thématique
0,refugies_en_russie,National (russe),Offre et recherche d'aide
1,refugies_en_russie_chat,National (russe),Générale
2,donbass_en_russie,National (russe),Générale
3,donbass_en_russie_chat,National (russe),Générale
4,navigateur,International,Générale
...,...,...,...
204,assist_khabarovsk_chat,Khabarovsk,Activités d'une organisation bénévole
205,ref_khabarovsk,Khabarovsk,Offre et recherche d'aide
206,assist_tcheliabinsk,Tcheliabinsk,Activités d'une organisation bénévole
207,pom_nt,Nizhny Tagil,Activités d'une organisation bénévole


In [ ]:
#make larger metadata with df info

df_nodes_base = df_meta.copy()

# filename: assist_sotchi.json -> Titre_support: assist_sotchi
df_nodes_base["Titre_support"] = df_nodes_base["filename"].str.replace(
    ".json", 
    "", 
    regex=False
)

# Добавляем Support_name, Ville/Région, Thématique из основной таблицы
df_nodes_base = df_nodes_base.merge(
    df_support,
    on="Titre_support",
    how="left"
)

df_nodes_base

,filename,json_name,json_type,json_id_raw,telegram_id,Titre_support,Ville/Région,Thématique
0,assist_sotchi.json,Помощь г Сочи Краснодарский край,public_supergroup,1700758525,channel1700758525,assist_sotchi,Sotchi,Activités d'une organisation bénévole
1,assist_krasnodar_chat.json,Краснодар. Помощь Chat,private_supergroup,1707130413,channel1707130413,assist_krasnodar_chat,Krasnodar et la région de Krasnodar,Activités d'une organisation bénévole
2,assistance_a_chebekino.json,Помощь Шебекино №1,public_channel,1673172889,channel1673172889,assistance_a_chebekino,Belgorod,Offre et recherche d'aide
3,kherson_in_tuapse.json,🍉 Херсонцы в Туапсе🍉,public_supergroup,1623635238,channel1623635238,kherson_in_tuapse,Krasnodar et la région de Krasnodar,Centres d'hébergement temporaire (PVR)
4,donbass_en_russie_chat.json,Донбасс в РФ📝ЧАТ,private_supergroup,1670808426,channel1670808426,donbass_en_russie_chat,National (russe),Générale
...,...,...,...,...,...,...,...,...
146,ref_penza.json,Беженцы Пенза РФ,public_supergroup,1709841251,channel1709841251,ref_penza,Penza,Activités d'une organisation bénévole
147,benevole_sidorova_chat.json,Чат Алла и Павел Сидоровы,public_supergroup,1632256033,channel1632256033,benevole_sidorova_chat,Oblast de Vladimir,Activités d'une organisation bénévole
148,ref_khabarovsk.json,Беженцы в Хабаровске,public_supergroup,1648385600,channel1648385600,ref_khabarovsk,Khabarovsk,Offre et recherche d'aide
149,interaction_assist_chat.json,Взаимодействие. Помощь беженцам в Старом Оскол...,public_supergroup,1625565768,channel1625565768,interaction_assist_chat,Stariy Oskol,Activités d'une organisation bénévole


In [39]:
df_nodes_base.to_csv("/Users/quentinnippert/Documents/mm_files/Telegram_network/metadata.csv", index=False, encoding="utf-8")

## 6. Construction de la table des nœuds pour Gephi

La table des nœuds transforme chaque espace Telegram en une entité du réseau. Elle contient un identifiant unique (`Id`), un label lisible pour l’affichage dans Gephi, ainsi que plusieurs attributs descriptifs : nom du fichier, type de l’espace, titre du support, région et thématique.

Cette étape vérifie également que les nœuds sans identifiant sont supprimés et que chaque identifiant apparaît une seule fois dans la table finale.


In [ ]:
df_gephi_nodes = pd.DataFrame({
    "Id": df_nodes_base["telegram_id"],
    
    # Это имя будет отображаться на графе
    "Label": df_nodes_base["Titre_support"].fillna(df_nodes_base["json_name"]),
    
    # Дополнительные атрибуты для фильтров / цветов в Gephi
    "Titre_support": df_nodes_base["Titre_support"],
    "json_name": df_nodes_base["json_name"],
    "json_type": df_nodes_base["json_type"],
    "Ville_Région": df_nodes_base["Ville/Région"],
    "Thématique": df_nodes_base["Thématique"],
    "filename": df_nodes_base["filename"]
})

# Убираем узлы без Id, если вдруг такие есть
df_gephi_nodes = df_gephi_nodes.dropna(subset=["Id"])

# На всякий случай убираем дубликаты Id
df_gephi_nodes = df_gephi_nodes.drop_duplicates(subset=["Id"])

df_gephi_nodes.head()

## 7. Construction de la table des liens

Les messages transférés extraits précédemment sont agrégés par couple source-cible. Le poids (`Weight`) d’un lien correspond au nombre de messages transférés observés entre deux espaces. La table est ensuite mise au format attendu par Gephi, avec les colonnes `Source`, `Target`, `Type` et `Weight`.

Le réseau est défini comme orienté (`Directed`), car le sens du transfert est important : il indique depuis quel espace un message provient et dans quel espace il a été relayé.


In [ ]:

df_gephi_edges = (
    df_edges_raw
    .groupby(["source_id", "target_id"])
    .size()
    .reset_index(name="Weight")
)

df_gephi_edges = df_gephi_edges.rename(columns={
    "source_id": "Source",
    "target_id": "Target"
})

# Для Gephi: Directed = направленная связь
df_gephi_edges["Type"] = "Directed"

# Порядок колонок
df_gephi_edges = df_gephi_edges[[
    "Source",
    "Target",
    "Type",
    "Weight"
]]

df_gephi_edges.head()

,Source,Target,Type,Weight
0,channel1197026404,channel1197026404,Directed,19
1,channel1197026404,channel1505469374,Directed,5
2,channel1197026404,channel1691851482,Directed,13
3,channel1240295821,channel1240295821,Directed,77
4,channel1240295821,channel1622616202,Directed,1


## 8. Suppression des boucles internes

Cette cellule produit une version de la table des liens sans *self-loops*, c’est-à-dire sans liens où la source et la cible correspondent au même espace. Cette version est utile pour Gephi lorsque l’analyse porte principalement sur les circulations entre espaces différents.


In [43]:
#delete self loops 

df_gephi_edges_no_selfloops = df_gephi_edges[
    df_gephi_edges["Source"] != df_gephi_edges["Target"]
].copy()

print(f"Edges before removing self-loops: {len(df_gephi_edges)}")
print(f"Edges after removing self-loops: {len(df_gephi_edges_no_selfloops)}")

Edges before removing self-loops: 362
Edges after removing self-loops: 323


## 9. Export des fichiers pour Gephi

Les tables finales sont sauvegardées au format CSV : une table des nœuds, une table des liens complète et une table des liens sans boucles internes. Ces fichiers peuvent ensuite être importés dans Gephi pour la visualisation et l’analyse du réseau.


In [44]:
df_gephi_nodes.to_csv("gephi_nodes.csv", index=False, encoding="utf-8-sig")
df_gephi_edges.to_csv("gephi_edges.csv", index=False, encoding="utf-8-sig")
df_gephi_edges_no_selfloops.to_csv("gephi_edges_no_selfloops.csv", index=False, encoding="utf-8-sig")

print("Files for Gephi saved:")
print("gephi_nodes.csv")
print("gephi_edges.csv")
print("gephi_edges_no_selfloops.csv")

Files for Gephi saved:
gephi_nodes.csv
gephi_edges.csv
gephi_edges_no_selfloops.csv


## 10. Contrôle de cohérence des nœuds et des liens

Avant l’import dans Gephi, cette vérification permet de s’assurer que toutes les sources et toutes les cibles présentes dans la table des liens existent bien dans la table des nœuds. Si des identifiants manquent, ils sont affichés afin de pouvoir corriger la table avant l’analyse visuelle du réseau.


In [45]:
node_ids = set(df_gephi_nodes["Id"])

missing_sources = set(df_gephi_edges["Source"]) - node_ids
missing_targets = set(df_gephi_edges["Target"]) - node_ids

print(f"Source without a node in nodes: {len(missing_sources)}")
print(f"Target without a node in nodes: {len(missing_targets)}")

print("Missing sources:", list(missing_sources)[:10])
print("Missing targets:", list(missing_targets)[:10])

Source without a node in nodes: 0
Target without a node in nodes: 0
Missing sources: []
Missing targets: []


## 11. Vérification rapide des fichiers exportés

La dernière partie recharge les fichiers CSV produits pour vérifier leur structure et afficher les premières lignes. Cette étape sert de contrôle final avant le passage à Gephi.


In [1]:
import pandas as pd 

df_nodes = pd.read_csv("gephi_nodes.csv")
df_edges = pd.read_csv("gephi_edges_no_selfloops.csv")

In [ ]:
df_nodes.head()

In [3]:
df_edges.head()

,Source,Target,Type,Weight
0,channel1197026404,channel1505469374,Directed,5
1,channel1197026404,channel1691851482,Directed,13
2,channel1240295821,channel1622616202,Directed,1
3,channel1240295821,channel1634791002,Directed,5
4,channel1240295821,channel1687942752,Directed,3
